**Required imports**

In [47]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import numpy as np

**Loading Dataset into pandas dataframe from hugging face**

In [48]:
ds = load_dataset("milistu/AMAZON-Products-2023")
x_train=pd.DataFrame(ds['train'].select_columns(['title', 'description', 'main_category']))
x_train.head()

,title,description,main_category
0,Anomie & Bonhomie,Amazon.com\nFans of Scritti Politti's synth-po...,Digital Music
1,Sunshine On My Shoulders: The Best Of John Den...,"“Sunshine On My Shoulders” is a 2CD, 36-track ...",Digital Music
2,18 Greatest Hits of 38 Special,Track Listings: 1. Rockin' Into The Night 2. ...,Digital Music
3,The Gift [CD],Second studio album by the multi-million-selli...,Digital Music
4,ΤΗΕ ΒΟΟΤLΕG SΕRΙΕS VοΙ. ᛐ7 ᛐ996-ᛐ997 FRΑԌΜΕΝΤՏ...,"EU Edition 2CD,\nDISC ONE - TIME OUT OF MIND [...",Digital Music


**Analyzing Data**

In [49]:
print(f"The number of rows and columns are : {x_train.shape}")
print(f"There are {x_train['main_category'].nunique()} unique categories")
print(f"\nNull values\n{x_train.isnull().sum()}")


The number of rows and columns are : (117243, 3)
There are 37 unique categories

Null values
title                0
description          0
main_category    24805
dtype: int64


In [50]:
print(f"\n{x_train['main_category'].value_counts().head(20)}")


main_category
AMAZON FASHION               36460
Amazon Home                  13874
Tools & Home Improvement      7945
Automotive                    5980
Cell Phones & Accessories     3471
Health & Personal Care        3416
All Electronics               3136
Sports & Outdoors             2942
Office Products               2925
Industrial & Scientific       2460
Pet Supplies                  1907
Computers                     1724
Digital Music                 1379
Handmade                      1127
Arts, Crafts & Sewing          874
Camera & Photo                 842
Musical Instruments            506
Appliances                     312
Home Audio & Theater           218
Video Games                    182
Name: count, dtype: int64


In [51]:
print(f"\n{x_train['main_category'].value_counts().tail(18)}")


main_category
Video Games                     182
Toys & Games                    146
Appstore for Android            142
Collectibles & Fine Art         114
Car Electronics                  69
All Beauty                       65
GPS & Navigation                 54
Portable Audio & Accessories     32
Collectible Coins                31
Grocery                          24
Baby                             23
Sports Collectibles              21
Software                         12
Entertainment                    11
Gift Cards                        8
Magazine Subscriptions            3
Amazon Devices                    2
Amazon Fire TV                    1
Name: count, dtype: int64


**Loading main_category predicted cleaned data**

In [52]:
cleaned_df=pd.read_csv('clean_data.csv')
cleaned_df.head()

,title,description,main_category
0,anomie bonhomie,amazoncom fan of scritti polittis synthpopfunk...,digital music
1,sunshine on my shoulder the best of john denve...,sunshine on my shoulder be a 2cd 36track colle...,digital music
2,18 greatest hit of 38 special,track listings 1 rockin into the night 2 hold ...,digital music
3,the gift cd,second studio album by the multimillionselling...,digital music
4,τηε βοοτlεg sεrιεs vοι ᛐ7 ᛐ996ᛐ997 frαԍμεντտ τ...,eu edition 2cd disc one time out of mind 2022 ...,digital music


In [53]:
cleaned_df['main_category'].value_counts().tail(20)

main_category
all electronics            4291
office products            3223
industrial & scientific    2996
pet supplies               2694
computers                  2307
handmade                   1959
digital music              1425
arts, crafts & sewing      1315
camera & photo              896
musical instruments         533
appliances                  434
video games                 252
home audio & theater        234
other                       203
toys & games                186
appstore for android        147
collectibles & fine art     116
all beauty                   84
car electronics              74
gps & navigation             60
Name: count, dtype: int64

analyzing and cleaning null data

In [65]:
print(cleaned_df.isnull().sum())
cleaned_df=cleaned_df.dropna()
print(f"\n\nAfter droping\n\n{cleaned_df.isna().sum()}")

title            0
description      0
main_category    0
dtype: int64


After droping

title            0
description      0
main_category    0
dtype: int64


**converting the text into vector**

In [56]:
t_tfidf=TfidfVectorizer(max_features=25000, ngram_range=(1,2), stop_words='english')
c_tfidf=TfidfVectorizer(max_features=50,stop_words='english')
d_tfidf=TfidfVectorizer(max_features=25000,ngram_range=(1,2),stop_words='english')

In [57]:
t_vector=t_tfidf.fit_transform(cleaned_df['title'])
c_vector=c_tfidf.fit_transform(cleaned_df['main_category'])
d_vector=d_tfidf.fit_transform(cleaned_df['description'])

assigning weight to each column based on their importance

In [58]:
tw=3.0
cw=2.0
dw=1.0

In [59]:
t_vector=t_vector*tw
c_vector=c_vector*cw
d_vector=d_vector*dw

final entire vectorized sparse matrix

In [60]:
s_v=sp.hstack([t_vector,c_vector,d_vector],format='csr')

**Function to calculate and return top similar product of query**

In [61]:
def similarity_calculate(title="",category="",description="",top_similar=5):
    title=title.lower()
    category=category.lower()
    description=description.lower()
    title_vector=t_tfidf.transform([title])*tw
    category_vector=c_tfidf.transform([category])*cw
    description_vector=d_tfidf.transform([description])*dw
    final_vector=sp.hstack([title_vector,category_vector,description_vector],format='csr')
    similarity=cosine_similarity(final_vector,s_v).flatten()
    top_similar_indx=np.argsort(similarity)[::-1][:top_similar]
    recommended = cleaned_df.iloc[top_similar_indx].copy()
    return recommended[['title','main_category']]

In [72]:
print(similarity_calculate("smart watch","electronic"))


                                                   title  \
46761                       nards smart watch dsgadgafdg   
45908                                 panhui smart watch   
37786                                    t12 smart watch   
45354  smart watch newest 185 tft hd display smart wa...   
37757                   lige smart watch charger for st5   

                   main_category  
46761            all electronics  
45908            all electronics  
37786  cell phones & accessories  
45354            all electronics  
37757            all electronics  
